# Foundation Models in Practice: From Language to Science with Hugging Face

**Lecture 3 — Live Demonstration and Guided Lab**  
**Presented by Dr. Fitsum Assamnew Andargie**  
**Duration:** 90 minutes  
**Environment:** Google Colab + Hugging Face

---

## Open this notebook in Google Colab

> ### [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fassamnew/Foundation-Models/blob/africa-workshop/Foundation_Models_in_Practice_Africa_Workshop.ipynb)
>
> ### → [Click here to open this notebook in Google Colab](https://colab.research.google.com/github/fassamnew/Foundation-Models/blob/africa-workshop/Foundation_Models_in_Practice_Africa_Workshop.ipynb)

Use the badge above **or** the **Click here** link. Both open the same Colab runtime for this workshop notebook.

**Repository:** [github.com/fassamnew/Foundation-Models/tree/africa-workshop](https://github.com/fassamnew/Foundation-Models/tree/africa-workshop)

---

> ### Central question
> **What makes a model a foundation model, and how can the same foundation-model idea be used across language, biology, vision, and Earth observation?**

## What you will do

During this guided lab, you will:

- inspect the compute available in Google Colab;
- load and use pretrained models from the Hugging Face Hub;
- open a Transformer to inspect tokens, representations, and attention;
- perform a small amount of real fine-tuning;
- move beyond language and use a **protein foundation model**;
- explore how foundation-model ideas extend to **multimodal and Earth-observation data**;
- discuss how African researchers and institutions can evaluate and adapt models responsibly.

## How to work through the notebook

For each exercise, use the following learning cycle:

**Predict → Run → Observe → Modify → Explain**

Before running a code cell, first ask yourself:

> **What do I expect this cell to do, and why?**

You do **not** need to be an expert Python programmer. The objective is to understand the ideas by interacting with real models.


# 0. Prepare the Colab Environment

## Why this section exists

Foundation models are not only algorithms. They are software systems that depend on:

- computational hardware,
- machine-learning frameworks,
- model architectures,
- learned parameters,
- and data.

Before loading any model, we will first inspect the environment in which the model will run.

### Connection to the main lecture

This section connects directly to:

**Section C — Hardware Enablement**

especially the discussion of GPUs, accelerators, memory, and the systems-engineering requirements of modern foundation models.

---

## Enable a GPU in Colab

A GPU makes model loading, fine-tuning, and generation much faster. Do this **before** running the install and setup cells below.

### Step 1 — Open the runtime settings

1. In the Colab menu bar at the top, click **Runtime**.
2. In the dropdown, click **Change runtime type**.

![Step 1: Runtime → Change runtime type](https://raw.githubusercontent.com/fassamnew/Foundation-Models/africa-workshop/images/colab-gpu-step1-runtime-menu.png)

### Step 2 — Select a GPU and save

1. In the **Change runtime type** dialog, keep **Runtime type** as **Python 3**.
2. Open the **Hardware accelerator** dropdown and choose a GPU option (often **T4 GPU**; any available GPU is fine).
3. Click **Save**.

Colab may reconnect the session after you save. That is expected.

![Step 2: Hardware accelerator → GPU, then Save](https://raw.githubusercontent.com/fassamnew/Foundation-Models/africa-workshop/images/colab-gpu-step2-change-runtime.png)

### Step 3 — Confirm the GPU is active

Check either of these:

- Top-right resource indicator: you should see **GPU** (not only RAM / Disk).
- Or open **Runtime → View resources** and confirm a GPU is listed.

Then run the accelerator-detection cell later in this section. You want output like:

```text
CUDA available: True
GPU: Tesla T4   # name may differ
```

If you see `CUDA available: False`, repeat Steps 1–2 and re-run the detection cell.

![Step 3: Confirm GPU is connected](https://raw.githubusercontent.com/fassamnew/Foundation-Models/africa-workshop/images/colab-gpu-step3-verify-gpu.png)

### If no GPU is available

Free Colab sometimes has no GPU capacity. You can still follow the notebook on CPU; later training cells already use smaller settings when a GPU is missing. Training and generation will be slower.


## Cell 0.1 — Install the software libraries

Google Colab already includes Python and PyTorch, but the exact versions of the Hugging Face libraries can vary between runtimes.

The following cell installs or updates the libraries used throughout the workshop.

### What the libraries do

- `transformers` — provides Transformer model architectures and pretrained-model interfaces.
- `datasets` — provides access to datasets used later in the fine-tuning exercise.
- `accelerate` — helps models use CPUs and GPUs efficiently.
- `huggingface_hub` — connects Python code to models and metadata on the Hugging Face Hub.
- `scikit-learn` — provides simple evaluation utilities.
- `matplotlib` — used later for visualizing model outputs.
- `pillow` — used for image handling in the multimodal section.

### Pillow version pin

We install **`pillow>=12.1.0`** on purpose. Pillow **12.0.0** can break imports on Colab with:

```text
ImportError: cannot import name '_Ink' from 'PIL._typing'
```

That bug was fixed in Pillow 12.1.0+.

If you still see a Pillow/`_Ink` import error after installing, choose **Runtime → Restart session**, then re-run from the import cell (you usually do not need to reinstall).

### What to expect

The installation output is intentionally quiet. A few package messages may appear.

> **Workshop note:** Run this cell once at the beginning of the session.


In [ ]:
# Pillow 12.0.0 breaks imports on Colab (_Ink missing). Use 12.1.0+.
%pip -q install -U transformers datasets accelerate huggingface_hub scikit-learn matplotlib "pillow>=12.1.0"

# If imports still fail after this cell, use Runtime → Restart session,
# then re-run from the next cell (you usually do not need to reinstall).


## Cell 0.2 — Import the libraries and make the experiment reproducible

Installing a library makes it available to Python. Importing it makes its functionality available inside this notebook.

We also set a **random seed**. Many machine-learning operations contain randomness. Using the same seed makes repeated runs more comparable.

### What to pay attention to

The cell prints the versions of the major software packages. These are useful for reproducibility: if code behaves differently on another machine, library versions are one of the first things to check.

In [ ]:
import gc
import platform
import random
import time

import numpy as np
import torch
import transformers
import datasets

from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("Environment ready.")
print(f"Python version:       {platform.python_version()}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version:     {datasets.__version__}")

# 1. What Compute Do We Have?

Before we load a model, let us inspect the hardware available to us.

Google Colab may provide either:

- a **CPU-only runtime**, or
- a runtime with a **GPU accelerator**.

The same Python model code may run on both, but training and large-model inference can be dramatically faster on a GPU.

### Before running the next cell

Predict:

1. Does your current Colab runtime have a GPU?
2. If it does, how much GPU memory is available?
3. Why might GPU memory matter when choosing a foundation model?

## Cell 1.1 — Detect the accelerator

This cell asks PyTorch whether CUDA-compatible GPU acceleration is available.

If a GPU is available, it also prints:

- the GPU model;
- total GPU memory;
- the device that later model code should use.

### Why this matters

A model's parameters and intermediate calculations must fit into available memory. As models become larger, memory becomes a major engineering constraint.

In [ ]:
cuda_available = torch.cuda.is_available()

print(f"CUDA / GPU available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"GPU model:            {gpu_name}")
    print(f"Total GPU memory:     {gpu_memory_gb:.2f} GB")

    DEVICE = torch.device("cuda")
else:
    print("No GPU detected. The notebook will run on CPU.")
    DEVICE = torch.device("cpu")

print(f"Selected device:      {DEVICE}")

## What just happened?

We still have **not loaded a foundation model**.

At this point we only have:

- a computer runtime;
- Python;
- PyTorch;
- the Hugging Face software libraries.

The model architecture and learned model parameters will arrive later.

This distinction is important because a working AI system is produced by several components working together. Hardware alone is not intelligence, and model weights alone cannot execute without a software and hardware environment.

## Cell 1.2 — Inspect currently used GPU memory

This small helper function lets us check GPU memory throughout the workshop.

Later, we will call it again after loading different models. This will make the relationship between **model size and hardware resources** visible.

### What to expect

At this stage, GPU memory use should be relatively small because no large model has been loaded.

In [ ]:
def show_gpu_memory():
    if not torch.cuda.is_available():
        print("GPU memory information is unavailable because this runtime is using the CPU.")
        return

    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"Allocated by PyTorch: {allocated:.2f} GB")
    print(f"Reserved by PyTorch:  {reserved:.2f} GB")
    print(f"Total GPU memory:     {total:.2f} GB")

show_gpu_memory()

## Cell 1.3 — A reusable cleanup function

During the workshop we will load several different models.

Google Colab GPUs have limited memory, so we will occasionally remove models that we no longer need and clear cached GPU memory.

The following helper function performs that cleanup.

You do not need to understand Python memory management in detail. The important systems lesson is:

> **Model selection is constrained not only by model capability, but also by the hardware on which the model must run.**

In [ ]:
def cleanup_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Unused Python objects and cached GPU memory have been cleared.")

## Discussion before moving on

Consider a hypothetical research team with access to a single modest GPU.

Would it always make sense for that team to choose the largest available foundation model?

Think about:

- memory;
- training time;
- inference time;
- electricity and infrastructure;
- number of users;
- cost;
- whether a smaller model already performs adequately for the task.

We will return to this question after loading and adapting real models.

### Main-lecture connection

The practical observations in this section support the lecture's discussion of:

- GPU enablement;
- custom accelerators;
- mixed-precision computation;
- memory-efficient model execution;
- compute access as an equity issue;
- systems engineering for foundation models.

# 2. Meet the Hugging Face Hub and Run a Pretrained Model

## Why this section exists

The lecture introduces foundation models as pretrained models that can later be reused or adapted for many tasks.

Before opening the internal architecture, we will first experience the simplest possible workflow:

1. identify a pretrained model;
2. inspect its metadata;
3. download it;
4. give it an input;
5. obtain a prediction.

This establishes the difference between **training a model** and **using an already trained model for inference**.

### Connection to the main lecture

This section connects mainly to:

- **Section B — Motivation & Definition**
- the shift from task-specific AI toward reusable pretrained models;
- the idea that foundation models become starting points for downstream applications.

> **Important:** The first model in this section is an English sentiment classifier. We use it only because it is small, fast, and easy to understand. It is **not** being presented as an Africa-ready language model. Later sections will examine multilingual representation and scientific foundation models.

## Cell 2.1 — Inspect a model before loading it

The Hugging Face Hub hosts model repositories. A repository can contain:

- model configuration;
- learned model parameters;
- tokenizer files;
- a model card;
- licensing information;
- tags describing tasks and frameworks;
- evaluation information and limitations.

Good practice is to inspect a model before executing it.

In this cell, we request metadata about a small DistilBERT model that has been fine-tuned for sentiment classification.

### What to pay attention to

Look for:

- the model identifier;
- the task;
- the license;
- the model's tags.

These are part of deciding whether a model is appropriate for a research or deployment setting.

In [ ]:
from huggingface_hub import model_info

SENTIMENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

info = model_info(SENTIMENT_MODEL)

print(f"Model ID:      {info.id}")
print(f"Pipeline task: {info.pipeline_tag}")
print(f"License:       {info.card_data.license if info.card_data else 'Not specified'}")
print(f"Downloads:     {info.downloads:,}" if info.downloads is not None else "Downloads: not available")

print("\nSelected tags:")
for tag in (info.tags or [])[:12]:
    print(" -", tag)

## What just happened?

We have inspected information about the model repository, but we still have **not performed inference**.

This distinction is important. A Hugging Face model repository is not simply a downloadable black box. It also carries documentation and metadata that should help users answer questions such as:

- What task was the model designed for?
- What data or benchmark was used?
- What language or domain does it target?
- What license applies?
- What known limitations are documented?

Later, when we examine African-language and scientific models, these questions become even more important.

## Cell 2.2 — Run the model through a high-level pipeline

The Hugging Face `pipeline` interface combines several steps into one high-level object.

For text classification, it performs approximately the following operations internally:

- prepare the input;
- tokenize it;
- convert it into tensors;
- execute the Transformer model;
- convert the model output into a human-readable label and score.

We begin with the high-level interface so that we can first see **what the model does**. Later, we will open the pipeline and inspect several of these steps ourselves.

### Before running the cell

Predict the likely labels for the three statements.

The statements deliberately come from different domains so that sentiment analysis is clearly treated as a demonstration task rather than the subject of the workshop.

In [ ]:
from transformers import pipeline

sentiment_classifier = pipeline(
    task="text-classification",
    model=SENTIMENT_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

example_statements = [
    "The new irrigation system performed much better than expected.",
    "The laboratory equipment failed repeatedly during the experiment.",
    "The rainfall measurements were collected successfully, although several sensors required recalibration."
]

for statement in example_statements:
    result = sentiment_classifier(statement)[0]

    print(f"Text:  {statement}")
    print(f"Label: {result['label']}")
    print(f"Score: {result['score']:.4f}")
    print("-" * 80)

## What should you observe?

The model returns two main pieces of information:

- a predicted sentiment label;
- a numerical score associated with that prediction.

The score is a model output, **not a guarantee that the prediction is correct**.

This is our first example of **inference**:

> We supplied new data to an already trained model and asked it to produce an output.

We did **not** update the model's learned parameters.

Later in the notebook, we will perform real training and see what changes.

## Cell 2.3 — Try your own input

Change the value of `my_text` and run the cell again.

Choose a sentence from any discipline represented in the room: agriculture, health, engineering, climate, biology, economics, education, or another area.

### Questions to consider

1. Does the output make sense?
2. What happens if the sentence contains both positive and negative information?
3. Would you trust this model in a high-stakes decision simply because its score is high?

In [ ]:
my_text = "The field trial produced promising results, but more testing is required."

prediction = sentiment_classifier(my_text)[0]

print("Input:", my_text)
print("Prediction:", prediction)

## Cell 2.4 — Check the memory footprint after loading the model

Earlier we measured GPU memory before loading any model.

Now call the same helper function again.

### Why this matters

This creates a concrete connection between the abstract idea of **model parameters** and the physical hardware required to execute them.

As we move to larger and more specialized models later, this relationship will become increasingly important.

In [ ]:
show_gpu_memory()

## Main idea from this exercise

At this point, you have used a pretrained model without training it.

Keep the following distinction in mind:

**Inference** means using learned parameters to make predictions on new inputs.

**Training** means updating learned parameters using data and an optimization procedure.

In later sections we will add a third concept:

**Fine-tuning** means starting from pretrained parameters and updating them for a more specific task.

# 3. How Does a Foundation Model Represent Language?

## Why this section exists

In the previous section, the Hugging Face `pipeline` hid most of the internal processing.

Now we begin to open that black box.

A Transformer does **not** directly receive a sentence as humans see it. Before the neural network can process language, the text must be converted into a sequence of numerical units.

This section introduces four important ideas:

- **tokenization** — dividing text into units the model knows how to process;
- **token IDs** — converting those units into integers;
- **attention masks** — telling the model which positions contain real input;
- **contextual representations** — vectors produced by the Transformer after it processes the sequence.

### Connection to the main lecture

This section connects directly to:

**Section D — Architecture & Data Representations**

especially:

- representation learning before Transformers;
- Transformer architecture;
- attention;
- self-supervised pretraining;
- representation gaps across languages.

We begin with language because the inputs are easy to inspect. Later, we will ask a more important question:

> **What if the tokens are not words at all?**

## Visual overview — From human text to model representation

The following cell creates a plain infographic showing the sequence of transformations we are about to inspect.

You do not need to memorize every term yet. We will open each stage one by one.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, 13)
ax.set_ylim(0, 3)
ax.axis("off")

steps = [
    (0.3, "Human text", '"AI can support\nagriculture"'),
    (2.8, "Tokenizer", "Break text into\nmodel units"),
    (5.3, "Token IDs", "[101, 9932, ...]"),
    (7.8, "Transformer", "Contextual\nprocessing"),
    (10.3, "Representations", "Vectors for each\ninput token"),
]

for x, title, body in steps:
    box = FancyBboxPatch(
        (x, 0.9), 2.0, 1.2,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.5,
        fill=False
    )
    ax.add_patch(box)
    ax.text(x + 1.0, 1.72, title, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(x + 1.0, 1.28, body, ha="center", va="center", fontsize=10)

for start_x in [2.3, 4.8, 7.3, 9.8]:
    arrow = FancyArrowPatch(
        (start_x, 1.5), (start_x + 0.45, 1.5),
        arrowstyle="->", mutation_scale=15, linewidth=1.4
    )
    ax.add_patch(arrow)

ax.set_title("How text becomes a Transformer representation", fontsize=14, pad=12)
plt.show()

## Cell 3.1 — Load a tokenizer without loading the full neural network

A tokenizer is a separate component from the Transformer itself.

This is important because we can inspect how text is represented **without downloading or executing the full model**.

We start with the tokenizer associated with the English DistilBERT model used earlier.

### What to expect

The tokenizer will be much smaller than the neural network weights. It contains the vocabulary and rules needed to convert text into model-readable units.

In [ ]:
from transformers import AutoTokenizer

ENGLISH_TOKENIZER_MODEL = "distilbert/distilbert-base-uncased"

english_tokenizer = AutoTokenizer.from_pretrained(
    ENGLISH_TOKENIZER_MODEL
)

print("Tokenizer loaded.")
print("Vocabulary size:", english_tokenizer.vocab_size)
print("Model maximum sequence length:", english_tokenizer.model_max_length)

## Cell 3.2 — Inspect tokens and token IDs for one sentence

The next cell shows the same sentence at three levels:

1. the original human-readable text;
2. the token pieces produced by the tokenizer;
3. the integer IDs corresponding to those pieces.

### What to pay attention to

Notice that:

- a token is not always the same thing as a word;
- special tokens may be added automatically;
- the neural network ultimately receives integers, not strings.

In [ ]:
example_text = "Foundation models learn reusable representations."

tokens = english_tokenizer.tokenize(example_text)
encoded = english_tokenizer(
    example_text,
    return_tensors="pt"
)

token_ids = encoded["input_ids"][0].tolist()
decoded_tokens = english_tokenizer.convert_ids_to_tokens(token_ids)

print("Original text:")
print(example_text)

print("\nTokenizer pieces:")
print(tokens)

print("\nInput IDs:")
print(token_ids)

print("\nTokens corresponding to the complete input IDs:")
print(decoded_tokens)

## What just happened?

The sentence was converted into a sequence from the model's fixed vocabulary.

This is the first important representation decision.

If a word is common in the tokenizer's vocabulary, it may be represented compactly. If it is uncommon, it may be split into several subword pieces. If the tokenizer cannot represent part of the text adequately, it may produce an unknown token.

This matters because **language coverage begins before the Transformer layers are even executed**.

# 3.3 An Africa-wide multilingual tokenizer experiment

The trainees in this workshop come from across Africa, so we should not examine representation through only one local language.

The next experiment uses a deliberately diverse set of languages and writing systems:

- English;
- French;
- Portuguese;
- Swahili;
- Hausa;
- Yoruba;
- isiZulu;
- Arabic;
- Amharic.

The example sentence has approximately the same meaning in each language: **artificial intelligence can support agriculture**.

> These sentences are used only as a tokenizer demonstration. They are not a language-quality benchmark.

We first apply the **English-oriented DistilBERT tokenizer** to all of them.

In [ ]:
african_language_examples = {
    "English": "Artificial intelligence can support agriculture.",
    "French": "L'intelligence artificielle peut soutenir l'agriculture.",
    "Portuguese": "A inteligência artificial pode apoiar a agricultura.",
    "Swahili": "Akili bandia inaweza kusaidia kilimo.",
    "Hausa": "Basirar wucin gadi na iya taimaka wa noma.",
    "Yoruba": "Ọgbọ́n atọwọda lè ṣe ìrànlọ́wọ́ fún iṣẹ́-ogbin.",
    "isiZulu": "Ubuhlakani bokwenziwa bungasiza ezolimo.",
    "Arabic": "يمكن للذكاء الاصطناعي دعم الزراعة.",
    "Amharic": "ሰው ሰራሽ አስተዋይነት ግብርናን መደገፍ ይችላል።",
}

unk_token = english_tokenizer.unk_token

for language, sentence in african_language_examples.items():
    pieces = english_tokenizer.tokenize(sentence)
    unk_count = pieces.count(unk_token)

    print(f"\n{language}")
    print("Text:", sentence)
    print("Tokens:", pieces)
    print("Number of tokens:", len(pieces))
    print("Unknown tokens:", unk_count)

## What should you observe?

Do not look only at the number of tokens.

Also look for:

- heavy fragmentation into many small pieces;
- `[UNK]` tokens;
- differences between Latin and non-Latin writing systems;
- whether diacritics appear to be represented cleanly;
- whether one language requires far more pieces than another for a sentence of similar meaning.

This is **not yet a test of model intelligence**.

We are examining only the input representation system.

The key question is:

> **Was this tokenizer designed from data that adequately represented the language we want to use?**

## Cell 3.4 — Compare the English tokenizer with a multilingual tokenizer

We now load the tokenizer from **XLM-RoBERTa**, a multilingual Transformer model trained for cross-lingual representation learning.

Importantly, we are loading only its tokenizer here, not the much larger neural network weights.

We will give the same African-language examples to both tokenizers and compare:

- token count;
- unknown-token count;
- degree of fragmentation.

### Before running the cell

Predict:

- Which languages do you expect the English-oriented tokenizer to handle poorly?
- Will the multilingual tokenizer make every language equally efficient?

In [ ]:
MULTILINGUAL_TOKENIZER_MODEL = "FacebookAI/xlm-roberta-base"

multilingual_tokenizer = AutoTokenizer.from_pretrained(
    MULTILINGUAL_TOKENIZER_MODEL
)

print("Multilingual tokenizer loaded.")
print("Vocabulary size:", multilingual_tokenizer.vocab_size)

In [ ]:
comparison_rows = []

for language, sentence in african_language_examples.items():
    english_pieces = english_tokenizer.tokenize(sentence)
    multilingual_pieces = multilingual_tokenizer.tokenize(sentence)

    english_unk = english_pieces.count(english_tokenizer.unk_token)
    multilingual_unk = multilingual_pieces.count(multilingual_tokenizer.unk_token)

    comparison_rows.append({
        "Language": language,
        "English-tokenizer tokens": len(english_pieces),
        "English-tokenizer UNKs": english_unk,
        "Multilingual tokens": len(multilingual_pieces),
        "Multilingual UNKs": multilingual_unk,
    })

import pandas as pd

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## Cell 3.5 — Inspect the actual token pieces side by side

Token counts alone can hide important details.

The next cell prints the token pieces from both tokenizers so that you can inspect the representation qualitatively.

Change `language_to_inspect` to any language in the dictionary.

In [ ]:
language_to_inspect = "Swahili"

sentence = african_language_examples[language_to_inspect]

print("Language:", language_to_inspect)
print("Text:", sentence)

print("\nEnglish-oriented DistilBERT tokenizer:")
print(english_tokenizer.tokenize(sentence))

print("\nMultilingual XLM-R tokenizer:")
print(multilingual_tokenizer.tokenize(sentence))

## Try it yourself

Change:

```python
language_to_inspect = "Swahili"
```

to another language such as:

- `"Hausa"`
- `"Yoruba"`
- `"isiZulu"`
- `"Arabic"`
- `"Amharic"`
- `"French"`
- `"Portuguese"`

If your own language is not included, add one sentence to `african_language_examples` and compare the tokenizers.

### Discussion questions

1. Does multilingual automatically mean equally good for every language?
2. What happens when a tokenizer breaks one language into far more pieces than another?
3. Could tokenization affect inference cost?
4. Could tokenization affect the amount of context that fits inside a model's maximum sequence length?
5. What additional evidence would we need before claiming that a model works well for a particular language?

# 3.6 What is an attention mask?

When multiple sequences are processed together, models often need to distinguish:

- positions containing real input tokens;
- positions added only for padding.

An **attention mask** communicates that distinction.

A value of:

- `1` usually means: **this is a real input position**;
- `0` usually means: **ignore this padding position**.

The following example intentionally processes two sentences of different lengths.

In [ ]:
batch = english_tokenizer(
    [
        "AI can support agriculture.",
        "Foundation models can be adapted for many downstream scientific applications."
    ],
    padding=True,
    return_tensors="pt"
)

print("Input IDs:")
print(batch["input_ids"])

print("\nAttention masks:")
print(batch["attention_mask"])

print("\nTensor shape:")
print(batch["input_ids"].shape)

## What should you notice?

Both sentences are stored in one rectangular tensor, so the shorter sentence must be padded.

The attention mask tells the model which positions are meaningful input and which positions were added only to make the batch rectangular.

This is a small implementation detail, but it becomes important when processing large datasets efficiently.

# 3.7 Load the Transformer itself

So far we have examined only tokenization.

Now we load the pretrained **DistilBERT encoder**.

Unlike the sentiment model used earlier, this base model does not include a task-specific sentiment classifier. Its main job is to transform the input sequence into **contextual vector representations**.

### What to expect

The model will return one representation vector for each input token.

In [ ]:
from transformers import AutoModel

encoder_model = AutoModel.from_pretrained(
    ENGLISH_TOKENIZER_MODEL
)

encoder_model = encoder_model.to(DEVICE)
encoder_model.eval()

print("Base Transformer loaded.")
print("Model class:", encoder_model.__class__.__name__)

## Cell 3.8 — Produce contextual representations

We now pass a tokenized sentence through the neural network.

The key output is called the **last hidden state**.

Its shape has three dimensions:

- batch size;
- number of token positions;
- hidden representation size.

The actual values are learned numerical representations rather than human-readable labels.

In [ ]:
representation_text = "Foundation models learn reusable representations."

representation_inputs = english_tokenizer(
    representation_text,
    return_tensors="pt"
)

representation_inputs = {
    key: value.to(DEVICE)
    for key, value in representation_inputs.items()
}

with torch.no_grad():
    representation_outputs = encoder_model(
        **representation_inputs,
        return_dict=True
    )

hidden_states = representation_outputs.last_hidden_state

print("Input text:")
print(representation_text)

print("\nHidden-state tensor shape:")
print(tuple(hidden_states.shape))

print("\nInterpretation:")
print("batch size =", hidden_states.shape[0])
print("token positions =", hidden_states.shape[1])
print("representation dimensions =", hidden_states.shape[2])

## What just happened?

The model transformed each token position into a high-dimensional vector.

These vectors are **contextual**.

That means the representation of a token is influenced by the other tokens in the sequence.

This is one of the key advances from earlier fixed word-vector approaches toward Transformer-based representation learning.

# 3.9 Context matters: the same word in two different sentences

Consider the word **bank**:

- `The bank approved the loan.`
- `We sat beside the bank of the river.`

The spelling is identical, but the meaning is different.

A contextual model should therefore produce different internal representations for the word depending on the surrounding sentence.

We will extract the representation of `bank` from each sentence and calculate their cosine similarity.

This is a pedagogical demonstration, not a full semantic evaluation.

In [ ]:
from torch.nn.functional import cosine_similarity

context_sentences = [
    "The bank approved the loan.",
    "We sat beside the bank of the river."
]

bank_vectors = []

for sentence in context_sentences:
    encoded_sentence = english_tokenizer(
        sentence,
        return_tensors="pt"
    )

    tokens = english_tokenizer.convert_ids_to_tokens(
        encoded_sentence["input_ids"][0]
    )

    bank_index = tokens.index("bank")

    encoded_sentence = {
        key: value.to(DEVICE)
        for key, value in encoded_sentence.items()
    }

    with torch.no_grad():
        outputs = encoder_model(
            **encoded_sentence,
            return_dict=True
        )

    bank_vector = outputs.last_hidden_state[0, bank_index, :]
    bank_vectors.append(bank_vector)

    print("\nSentence:", sentence)
    print("Tokens:", tokens)
    print("Position of 'bank':", bank_index)

similarity = cosine_similarity(
    bank_vectors[0].unsqueeze(0),
    bank_vectors[1].unsqueeze(0)
).item()

print("\nCosine similarity between the two contextual 'bank' vectors:")
print(round(similarity, 4))

## Interpretation

The two vectors are not expected to be identical because the surrounding context is different.

The important lesson is not the exact similarity number.

The important lesson is:

> **A Transformer representation is shaped by context.**

This prepares us for the next concept: **self-attention**, one of the mechanisms through which information from different positions can influence each other.

### Main-lecture connection

This section makes several ideas from **Section D — Architecture & Data Representations** concrete:

- tokenization is part of representation;
- language coverage can already differ at the tokenizer level;
- model inputs are numerical IDs;
- Transformers produce contextual vector representations;
- the same surface word can receive different internal representations depending on context.

In the next section, we inspect the attention mechanism that helps create those contextual representations.

# 4. How Does Self-Attention Connect Information Across a Sequence?

## Why this section exists

In the previous section, we observed that the word `bank` receives different internal representations in different contexts.

How can surrounding words influence the representation of one token?

One of the central mechanisms is **self-attention**.

Self-attention allows each token position to assign different weights to other positions in the same sequence while constructing a new representation.

### Connection to the main lecture

This section connects directly to:

- **Bahdanau, Cho & Bengio (2014)** — attention in neural machine translation;
- **Vaswani et al. (2017)** — *Attention Is All You Need*;
- **Section D — Details of the attention mechanism and Transformer architecture**.

The objective here is not to derive every equation. The objective is to inspect a real attention tensor and understand what its dimensions mean.

## Visual overview — What self-attention does

The following plain infographic shows the conceptual role of self-attention.

Each input token begins with a representation. During self-attention, the model compares token positions and calculates weighted combinations of information from the sequence.

The result is an updated representation for each position.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12.5, 4))
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 4)
ax.axis("off")

left_tokens = ["crop", "disease", "affects", "maize"]
right_tokens = ["crop′", "disease′", "affects′", "maize′"]

for i, token in enumerate(left_tokens):
    y = 3.25 - i * 0.8
    box = FancyBboxPatch((0.5, y - 0.25), 1.5, 0.5,
                         boxstyle="round,pad=0.03",
                         linewidth=1.3, fill=False)
    ax.add_patch(box)
    ax.text(1.25, y, token, ha="center", va="center", fontsize=10)

center = FancyBboxPatch((4.4, 1.0), 3.2, 2.0,
                        boxstyle="round,pad=0.05",
                        linewidth=1.6, fill=False)
ax.add_patch(center)
ax.text(6.0, 2.35, "SELF-ATTENTION", ha="center", va="center",
        fontsize=12, fontweight="bold")
ax.text(6.0, 1.85, "Compare token positions", ha="center", fontsize=10)
ax.text(6.0, 1.48, "Assign attention weights", ha="center", fontsize=10)
ax.text(6.0, 1.12, "Mix contextual information", ha="center", fontsize=10)

for i, token in enumerate(right_tokens):
    y = 3.25 - i * 0.8
    box = FancyBboxPatch((10.3, y - 0.25), 1.5, 0.5,
                         boxstyle="round,pad=0.03",
                         linewidth=1.3, fill=False)
    ax.add_patch(box)
    ax.text(11.05, y, token, ha="center", va="center", fontsize=10)

ax.add_patch(FancyArrowPatch((2.1, 2.0), (4.3, 2.0),
                             arrowstyle="->", mutation_scale=16, linewidth=1.4))
ax.add_patch(FancyArrowPatch((7.7, 2.0), (10.2, 2.0),
                             arrowstyle="->", mutation_scale=16, linewidth=1.4))

ax.text(1.25, 3.75, "Input representations", ha="center",
        fontsize=11, fontweight="bold")
ax.text(11.05, 3.75, "Contextualized representations", ha="center",
        fontsize=11, fontweight="bold")

ax.set_title("Self-attention lets each position use information from other positions",
             fontsize=14, pad=10)

plt.show()

## Cell 4.1 — Ask DistilBERT to return its attention weights

By default, many model calls return only the final hidden representations.

Here we explicitly request the attention tensors.

For a Transformer layer, the attention tensor has four dimensions:

1. batch;
2. attention head;
3. query-token position;
4. key-token position.

DistilBERT uses multiple attention heads in each Transformer layer. Different heads can learn different patterns of interaction.

### What to expect

The cell will print:

- how many Transformer layers returned attention;
- the shape of one layer's attention tensor.

In [ ]:
attention_text = "Climate variability affects crop production across regions."

attention_inputs = english_tokenizer(
    attention_text,
    return_tensors="pt"
)

attention_inputs = {
    key: value.to(DEVICE)
    for key, value in attention_inputs.items()
}

# Use eager attention so attention weights are explicitly available
# across current Transformers attention backends.
attention_model = AutoModel.from_pretrained(
    ENGLISH_TOKENIZER_MODEL,
    attn_implementation="eager"
).to(DEVICE)

attention_model.eval()

with torch.no_grad():
    attention_outputs = attention_model(
        **attention_inputs,
        output_attentions=True,
        return_dict=True
    )

attentions = attention_outputs.attentions

print("Number of Transformer layers with attention:", len(attentions))
print("Shape of one attention tensor:", tuple(attentions[0].shape))

print("\nInterpretation:")
print("batch size =", attentions[0].shape[0])
print("attention heads =", attentions[0].shape[1])
print("query-token positions =", attentions[0].shape[2])
print("key-token positions =", attentions[0].shape[3])

## What just happened?

For every Transformer layer, the model returned an attention matrix for every attention head.

If the input has `N` token positions, each attention head contains an approximately `N × N` matrix.

A row answers a question similar to:

> **When updating this token position, how is attention distributed over the available token positions?**

The values in each row are normalized attention weights.

## Cell 4.2 — See the tokens represented by the matrix

Before drawing the attention matrix, we need to know which token belongs to each row and column.

Remember that the tokenizer may have inserted special tokens.

In [ ]:
attention_tokens = english_tokenizer.convert_ids_to_tokens(
    attention_inputs["input_ids"][0]
)

print("Text:")
print(attention_text)

print("\nToken positions:")
for position, token in enumerate(attention_tokens):
    print(f"{position:>2}: {token}")

## Cell 4.3 — Visualize one attention head

We now visualize a single attention head from a single Transformer layer.

Each row corresponds to a **query token**.

Each column corresponds to a **token being attended to**.

Larger values indicate greater attention weight for that particular head and layer.

### Important caution

An attention map is useful for understanding model mechanics, but it is **not automatically an explanation of why a model made a final decision**.

Do not interpret a high attention value as proof of causal importance.

In [ ]:
layer_to_view = 0
head_to_view = 0

attention_matrix = (
    attentions[layer_to_view][0, head_to_view]
    .detach()
    .cpu()
    .numpy()
)

plt.figure(figsize=(9, 7))
plt.imshow(attention_matrix, aspect="auto")
plt.xticks(
    range(len(attention_tokens)),
    attention_tokens,
    rotation=70
)
plt.yticks(
    range(len(attention_tokens)),
    attention_tokens
)
plt.xlabel("Key token — position being attended to")
plt.ylabel("Query token — position being updated")
plt.title(
    f"DistilBERT attention — layer {layer_to_view}, head {head_to_view}"
)
plt.colorbar(label="Attention weight")
plt.tight_layout()
plt.show()

## Cell 4.4 — Average attention across all heads in one layer

One head shows only one learned attention pattern.

For a simpler high-level visualization, we can average the attention matrices across all heads within a selected layer.

This destroys information about individual head specialization, but it is useful for seeing the overall structure of the tensor.

In [ ]:
layer_to_average = len(attentions) - 1

mean_attention = (
    attentions[layer_to_average][0]
    .mean(dim=0)
    .detach()
    .cpu()
    .numpy()
)

plt.figure(figsize=(9, 7))
plt.imshow(mean_attention, aspect="auto")
plt.xticks(
    range(len(attention_tokens)),
    attention_tokens,
    rotation=70
)
plt.yticks(
    range(len(attention_tokens)),
    attention_tokens
)
plt.xlabel("Key token")
plt.ylabel("Query token")
plt.title(
    f"Mean attention across heads — layer {layer_to_average}"
)
plt.colorbar(label="Mean attention weight")
plt.tight_layout()
plt.show()

## Cell 4.5 — Try another sentence, layer, or attention head

Modify the variables below and re-run the attention cells.

Possible sentences:

- `Drought affects food production and water availability.`
- `A mutation can change the function of a protein.`
- `Satellite observations can reveal changes in vegetation.`
- a sentence from your own research field.

Also experiment with different:

- `layer_to_view`;
- `head_to_view`.

### Questions to discuss

1. Do different heads show identical patterns?
2. Do different Transformer layers show identical patterns?
3. Why might a model benefit from multiple attention heads?
4. Why should we be cautious about treating attention as an explanation?

## Main idea from this section

Self-attention provides a mechanism through which token positions can exchange information while new contextual representations are built.

This helps explain the result we saw earlier:

the same surface word can receive different contextual representations depending on the sequence around it.

In the main lecture, this connects the historical development from recurrent sequence models and early attention mechanisms to the Transformer architecture introduced by Vaswani and colleagues.

### Next step

We now understand:

- input representation;
- tokenization;
- contextual vectors;
- self-attention.

The next question is:

> **Where did all of these useful model parameters come from?**

To answer that, we will demonstrate **self-supervised pretraining intuition and fine-tuning**.

# 5. Where Did the Useful Parameters Come From? Pretraining and Fine-Tuning

## Why this section exists

So far, we have used a model whose parameters were already learned.

That raises an important question:

> **How did a general pretrained model become useful before we ever gave it our task-specific dataset?**

Foundation models are typically first trained on large-scale data using objectives that allow the model to learn reusable representations.

After that, the model can be:

- used directly;
- prompted;
- fine-tuned;
- adapted using parameter-efficient methods;
- incorporated into a larger system.

In this section we will first demonstrate the intuition behind **self-supervised pretraining**, and then perform a small amount of real **fine-tuning**.

### Connection to the main lecture

This section connects directly to:

- **Section B — What is a foundation model?**
- **Section D — Self-supervised pretraining objectives**
- **Section C — Compute requirements and mixed precision**
- the distinction between pretraining a foundation model and adapting one.

## Visual overview — Pretraining is not the same as fine-tuning

The next cell creates a plain infographic that distinguishes three stages:

1. large-scale pretraining;
2. the reusable pretrained model;
3. smaller downstream adaptation.

The exercise we perform in this workshop is the **third stage**, not the first.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12.5, 4.2))
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 4.2)
ax.axis("off")

boxes = [
    (0.5, 1.25, 3.0, 1.7,
     "LARGE-SCALE PRETRAINING",
     "Broad data\nSelf-supervised objective\nLarge compute"),
    (4.75, 1.25, 3.0, 1.7,
     "PRETRAINED MODEL",
     "Reusable parameters\nGeneral representations\nStarting point"),
    (9.0, 1.25, 3.0, 1.7,
     "DOWNSTREAM ADAPTATION",
     "Smaller task dataset\nFine-tuning / adapters\nTask-specific output"),
]

for x, y, w, h, title, body in boxes:
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.05,rounding_size=0.08",
        linewidth=1.5,
        fill=False
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + 1.22, title,
            ha="center", va="center",
            fontsize=11, fontweight="bold")
    ax.text(x + w/2, y + 0.62, body,
            ha="center", va="center", fontsize=10)

ax.add_patch(FancyArrowPatch((3.55, 2.1), (4.65, 2.1),
                             arrowstyle="->", mutation_scale=17, linewidth=1.4))
ax.add_patch(FancyArrowPatch((7.8, 2.1), (8.9, 2.1),
                             arrowstyle="->", mutation_scale=17, linewidth=1.4))

ax.text(6.25, 3.75,
        "The workshop reuses pretraining; it does not reproduce foundation-model pretraining",
        ha="center", va="center", fontsize=13, fontweight="bold")

plt.show()

# 5.1 Self-supervised pretraining intuition

BERT-family models use **masked language modeling** as a pretraining objective.

The idea is simple:

- hide part of an input sequence;
- ask the model to predict what belongs in the missing position;
- repeat this across very large amounts of text.

The labels are created from the text itself, which is why the approach is called **self-supervised**.

We will use the base DistilBERT model to predict a masked token.

## Cell 5.1 — Load the masked-language-model version of DistilBERT

The model in this cell has a head designed to predict vocabulary tokens.

This is different from the sentiment-classification head we used earlier.

In [ ]:
from transformers import pipeline

fill_mask_model = pipeline(
    task="fill-mask",
    model=ENGLISH_TOKENIZER_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

print("Masked-language model ready.")

## Cell 5.2 — Predict a missing token

The tokenizer associated with DistilBERT uses `[MASK]` to represent the hidden position.

Before running the cell, predict several words that could reasonably fill the blank.

In [ ]:
masked_examples = [
    "Machine learning can help researchers [MASK] patterns in data.",
    "Satellite observations can help scientists monitor [MASK]."
]

for sentence in masked_examples:
    print("\nInput:", sentence)

    predictions = fill_mask_model(sentence, top_k=5)

    for item in predictions:
        print(
            f"{item['token_str']!r:15s} "
            f"score={item['score']:.4f}"
        )

## What just happened?

The model used its learned representations to estimate plausible tokens from context.

The exact workshop examples are small, but the underlying pretraining principle scales:

the model repeatedly learns from relationships within the data itself.

This is the important concept to remember, because later we will encounter a **protein model trained with a closely related masked-token idea**.

At that point, the missing token will not be an English word.

It will be an **amino acid**.

# 5.2 Fine-tuning: adapt a pretrained model with a small labeled dataset

Now we perform actual training.

We will use the SST-2 sentiment dataset because it is:

- small;
- well supported;
- easy to understand;
- fast enough for a live Colab exercise.

The scientific importance of this section is **not sentiment analysis**.

Sentiment is simply the task we use to make the mechanics of adaptation visible.

### Our controlled workshop experiment

To keep runtime short:

- GPU runtime: use 600 training examples and 200 validation examples;
- CPU runtime: use a smaller subset;
- use only one training epoch;
- freeze the embeddings and lower Transformer layers;
- update the upper layers and task-specific classifier.

This is real gradient-based training, but it is intentionally small enough for a workshop.

## Cell 5.3 — Load a small labeled dataset

SST-2 contains sentences labeled:

- `0` — negative;
- `1` — positive.

We select a deterministic subset so participants work with approximately the same examples.

In [ ]:
from datasets import load_dataset

if torch.cuda.is_available():
    TRAIN_EXAMPLES = 600
    VALIDATION_EXAMPLES = 200
else:
    TRAIN_EXAMPLES = 160
    VALIDATION_EXAMPLES = 80

full_train_dataset = load_dataset(
    "nyu-mll/glue",
    "sst2",
    split="train"
)

full_validation_dataset = load_dataset(
    "nyu-mll/glue",
    "sst2",
    split="validation"
)

train_dataset = (
    full_train_dataset
    .shuffle(seed=SEED)
    .select(range(TRAIN_EXAMPLES))
)

validation_dataset = (
    full_validation_dataset
    .shuffle(seed=SEED)
    .select(range(VALIDATION_EXAMPLES))
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

print("\nOne example:")
print(train_dataset[0])

print("\nLabels:")
print("0 = negative")
print("1 = positive")

## Cell 5.4 — Tokenize the training data

The model cannot train directly on raw strings.

We therefore apply the same tokenizer to every sentence in the dataset.

We also limit sequence length to keep this workshop experiment computationally small.

In [ ]:
MAX_LENGTH = 96

def tokenize_training_batch(batch):
    return english_tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train_dataset = train_dataset.map(
    tokenize_training_batch,
    batched=True
)

tokenized_validation_dataset = validation_dataset.map(
    tokenize_training_batch,
    batched=True
)

print("Tokenization complete.")
print(tokenized_train_dataset[0])

## Cell 5.5 — Create a classification model from pretrained DistilBERT

We now load pretrained DistilBERT parameters and attach a two-class classification head.

The pretrained encoder already contains learned language representations.

The classification head is task-specific.

For workshop speed, we freeze:

- the embedding layer;
- the first four of DistilBERT's six Transformer layers.

The final two Transformer layers and classification components remain trainable.

### What to pay attention to

We will count:

- total parameters;
- trainable parameters;
- the fraction actually being updated.

This is another direct connection between algorithm design and compute requirements.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

training_model = AutoModelForSequenceClassification.from_pretrained(
    ENGLISH_TOKENIZER_MODEL,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

# Freeze embeddings.
training_model.distilbert.embeddings.requires_grad_(False)

# Freeze the first four Transformer layers.
for layer in training_model.distilbert.transformer.layer[:4]:
    layer.requires_grad_(False)

total_parameters = sum(
    parameter.numel()
    for parameter in training_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in training_model.parameters()
    if parameter.requires_grad
)

print(f"Total parameters:     {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(
    f"Trainable fraction:   "
    f"{100 * trainable_parameters / total_parameters:.2f}%"
)

data_collator = DataCollatorWithPadding(
    tokenizer=english_tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

## What should you understand from the parameter counts?

A pretrained model may contain tens of millions, hundreds of millions, or billions of parameters.

Fine-tuning does not necessarily require updating all of them.

Freezing some parameters reduces:

- gradient computation;
- optimizer state;
- memory use;
- training time.

Modern parameter-efficient methods take this idea even further by training relatively small additional parameter sets while leaving most pretrained weights unchanged.

We will discuss methods such as LoRA later, but we will not make them part of the core 90-minute exercise.

## Cell 5.6 — Define the training procedure

Hugging Face `Trainer` manages the optimization loop for us.

We specify:

- learning rate;
- batch size;
- one training epoch;
- when to evaluate;
- whether to use mixed-precision FP16 on a GPU.

The model still performs ordinary gradient-based optimization. `Trainer` simply organizes the training loop.

In [ ]:
from transformers import Trainer, TrainingArguments

def compute_accuracy(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    accuracy = (predictions == labels).mean()

    return {
        "accuracy": float(accuracy)
    }

training_arguments = TrainingArguments(
    output_dir="./workshop_distilbert_sst2",
    learning_rate=5e-5,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 8,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 16,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

trainer = Trainer(
    model=training_model,
    args=training_arguments,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    data_collator=data_collator,
    processing_class=english_tokenizer,
    compute_metrics=compute_accuracy
)

print("Trainer configured.")

## Cell 5.7 — Measure performance before training

The classification head has just been created for this task.

Before optimization, we establish a baseline.

### Predict before running

For a roughly balanced two-class problem, what accuracy would random guessing produce on average?

Do not expect the exact baseline here to be exactly 50%, because:

- the validation subset may not be perfectly balanced;
- the newly initialized classifier is random.

In [ ]:
baseline_metrics = trainer.evaluate()

print("Before fine-tuning:")
print(
    "Validation accuracy:",
    round(baseline_metrics["eval_accuracy"], 3)
)
print(
    "Validation loss:",
    round(baseline_metrics["eval_loss"], 3)
)

# Cell 5.8 — Train the model

This cell performs the actual fine-tuning.

We record the wall-clock time so the workshop can directly observe the computational cost of this small experiment.

### While the model is training

Do not simply watch the progress bar.

Discuss:

1. Which parameters are being updated?
2. Which parameters came from pretraining?
3. Why can a pretrained model adapt using a relatively small labeled dataset?
4. How would training change if the model had billions of parameters?
5. Why do hardware, precision, distributed training, and memory optimization become important at scale?

In [ ]:
training_start_time = time.perf_counter()

training_result = trainer.train()

training_elapsed_seconds = (
    time.perf_counter() - training_start_time
)

print(
    f"\nFine-tuning wall-clock time: "
    f"{training_elapsed_seconds:.1f} seconds"
)

print(
    f"Fine-tuning wall-clock time: "
    f"{training_elapsed_seconds / 60:.2f} minutes"
)

## Cell 5.9 — Evaluate after fine-tuning

Now evaluate the same validation subset again.

The exact accuracy is not the main lesson.

What matters is whether the task-specific model improved after a small amount of adaptation.

In [ ]:
fine_tuned_metrics = trainer.evaluate()

print("Before fine-tuning accuracy:",
      round(baseline_metrics["eval_accuracy"], 3))

print("After fine-tuning accuracy:",
      round(fine_tuned_metrics["eval_accuracy"], 3))

print("After fine-tuning loss:",
      round(fine_tuned_metrics["eval_loss"], 3))

## Cell 5.10 — Compare the result visually

The following simple chart compares validation accuracy before and after the workshop fine-tuning run.

Your exact values may differ from another participant's values.

In [ ]:
before_accuracy = baseline_metrics["eval_accuracy"]
after_accuracy = fine_tuned_metrics["eval_accuracy"]

plt.figure(figsize=(6, 4))
plt.bar(
    ["Before fine-tuning", "After fine-tuning"],
    [before_accuracy, after_accuracy]
)
plt.ylim(0, 1)
plt.ylabel("Validation accuracy")
plt.title("Effect of a small fine-tuning experiment")
plt.tight_layout()
plt.show()

## What did we actually train?

This is one of the most important distinctions in the workshop.

We **did not train a foundation model from scratch**.

Instead:

- large-scale pretraining happened before the workshop;
- we downloaded the resulting pretrained parameters;
- we attached a task-specific classifier;
- we updated a subset of parameters using a small labeled dataset.

That is **adaptation**.

### Main-lecture connection

This exercise provides a practical interpretation of several lecture concepts:

- self-supervised pretraining creates reusable representations;
- downstream tasks can reuse those representations;
- training requires compute and memory;
- mixed precision can reduce some compute/memory costs;
- access to pretrained models dramatically changes what smaller research teams can build;
- adapting an existing model is very different from financing and operating frontier-scale pretraining.

### The key transition

Up to this point, every input has been natural language.

Now we make the most important conceptual move in the notebook:

> **Does a foundation model require words?**

The answer is no.

In the next section, we will use a Transformer whose “language” is made of **amino acids**.

# 6. Beyond Language: A Protein Foundation Model with ESM-2

## Why this section exists

This is the conceptual turning point of the workshop.

Everything so far could leave the impression that a foundation model is mainly a language model.

It is not.

A foundation model is a broader approach:

- train on large-scale domain data;
- learn reusable representations;
- reuse or adapt those representations for downstream tasks.

In biology, a protein can be represented as a sequence of amino acids.

A Transformer can therefore learn relationships within protein sequences in a way that is conceptually related to how language models learn relationships within text.

We will use **ESM-2**, a protein language model from Meta AI.

For workshop reliability, we use the smallest ESM-2 checkpoint:

`facebook/esm2_t6_8M_UR50D`

It contains roughly 8 million parameters, making it practical for Colab while preserving the core scientific idea.

### Connection to the main lecture

This section connects directly to:

**Section E — Domain-Leading Models: Biology and Medicine**

and especially:

- ESM-2;
- self-supervised learning beyond human language;
- reusable biological representations;
- adaptation of foundation models to scientific tasks.

## Visual overview — The Transformer idea crosses domains

The next plain infographic compares the language example you already understand with the protein example we are about to run.

The important idea is that the Transformer does not require its tokens to be words.

The domain determines what the tokens represent.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12.5, 5.3))
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 5.3)
ax.axis("off")

# Language row
language_boxes = [
    (0.4, 3.2, 2.6, 1.0, "TEXT DATA", "sentences"),
    (3.6, 3.2, 2.6, 1.0, "TOKENS", "words / subwords"),
    (6.8, 3.2, 2.6, 1.0, "TRANSFORMER", "context learning"),
    (10.0, 3.2, 2.1, 1.0, "REPRESENTATION", "vectors"),
]

# Protein row
protein_boxes = [
    (0.4, 1.0, 2.6, 1.0, "PROTEIN DATA", "amino-acid sequences"),
    (3.6, 1.0, 2.6, 1.0, "TOKENS", "amino acids"),
    (6.8, 1.0, 2.6, 1.0, "TRANSFORMER", "context learning"),
    (10.0, 1.0, 2.1, 1.0, "REPRESENTATION", "vectors"),
]

for row in [language_boxes, protein_boxes]:
    for x, y, w, h, title, body in row:
        box = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.04,rounding_size=0.07",
            linewidth=1.4,
            fill=False
        )
        ax.add_patch(box)
        ax.text(x + w/2, y + 0.66, title,
                ha="center", va="center",
                fontsize=10.5, fontweight="bold")
        ax.text(x + w/2, y + 0.30, body,
                ha="center", va="center", fontsize=9.5)

for y in [3.7, 1.5]:
    for start, end in [(3.05, 3.55), (6.25, 6.75), (9.45, 9.95)]:
        ax.add_patch(
            FancyArrowPatch(
                (start, y), (end, y),
                arrowstyle="->", mutation_scale=15, linewidth=1.3
            )
        )

ax.text(6.25, 4.85,
        "One foundation-model principle, different scientific data",
        ha="center", va="center", fontsize=14, fontweight="bold")

ax.text(0.15, 3.7, "Language", rotation=90,
        va="center", ha="center", fontsize=10, fontweight="bold")
ax.text(0.15, 1.5, "Biology", rotation=90,
        va="center", ha="center", fontsize=10, fontweight="bold")

plt.show()

# 6.1 Release the previous training models

The earlier sections intentionally kept several DistilBERT objects in memory.

Before loading a new scientific model, we remove objects that are no longer required.

This is another practical systems-engineering habit when working in a memory-limited notebook environment.

In [ ]:
objects_to_release = [
    "trainer",
    "training_model",
    "attention_model",
    "encoder_model",
    "fill_mask_model",
    "sentiment_classifier"
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

cleanup_memory()
show_gpu_memory()

# 6.2 Inspect the ESM-2 model before running it

As we did with DistilBERT, we first inspect the model repository metadata.

### What to look for

Notice that the model is associated with:

- the `fill-mask` task;
- the ESM architecture;
- protein sequence modeling rather than ordinary text;
- an open model license.

The model card explains that ESM-2 was trained using a masked-language-modeling objective on protein sequences and can be fine-tuned for downstream protein tasks.

In [ ]:
from huggingface_hub import model_info

ESM_MODEL = "facebook/esm2_t6_8M_UR50D"

esm_info = model_info(ESM_MODEL)

print("Model ID:", esm_info.id)
print("Pipeline task:", esm_info.pipeline_tag)
print("License:", esm_info.card_data.license if esm_info.card_data else "Not specified")

print("\nSelected tags:")
for tag in (esm_info.tags or [])[:12]:
    print(" -", tag)

# 6.3 Inspect the protein tokenizer

Now we load only the ESM-2 tokenizer.

Unlike the DistilBERT tokenizer, its basic vocabulary is built around amino-acid symbols and a small number of special tokens.

### Before running

Predict:

- Will the vocabulary need tens of thousands of entries?
- How many standard amino acids are commonly represented in protein sequences?

In [ ]:
from transformers import AutoTokenizer

esm_tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL)

print("ESM tokenizer loaded.")
print("Vocabulary size:", esm_tokenizer.vocab_size)
print("Mask token:", esm_tokenizer.mask_token)
print("Mask token ID:", esm_tokenizer.mask_token_id)

print("\nVocabulary:")
print(esm_tokenizer.get_vocab())

## What should you notice?

The vocabulary is dramatically smaller than a natural-language tokenizer.

That makes sense.

Human languages contain enormous lexical variation. Protein sequences are constructed from a much smaller amino-acid alphabet plus special symbols.

But a small vocabulary does **not** mean the scientific problem is simple.

The challenge is learning complex relationships among positions across very large collections of biological sequences.

# 6.4 The same self-supervised idea: predict a masked amino acid

Earlier, DistilBERT predicted a missing word or subword.

Now ESM-2 will predict a missing amino acid.

The example sequence below comes from the example associated with the ESM-2 model card.

The `<mask>` token marks the hidden amino-acid position.

### Before running

Do not worry if you cannot predict the amino acid yourself.

The important question is:

> **Is the computational objective conceptually familiar?**

In [ ]:
from transformers import pipeline

protein_fill_mask = pipeline(
    task="fill-mask",
    model=ESM_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

masked_protein = (
    "MQIFVKTLTGKTITLEVEPS<mask>"
    "TIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)

protein_predictions = protein_fill_mask(
    masked_protein,
    top_k=5
)

print("Masked protein sequence:")
print(masked_protein)

print("\nTop predictions for the missing amino acid:")

for prediction in protein_predictions:
    print(
        f"{prediction['token_str']!r:8s} "
        f"score={prediction['score']:.4f}"
    )

## What just happened?

This is the key scientific connection.

Earlier:

- data = natural-language text;
- token = word or subword;
- objective = predict a masked language token.

Now:

- data = protein sequence;
- token = amino acid;
- objective = predict a masked amino-acid token.

The domains are different, but the **self-supervised representation-learning idea is closely related**.

This is why the term *language model* can sometimes be used metaphorically in scientific sequence modeling: the model learns statistical structure in sequences, even though the “language” is biological rather than human.

# 6.5 Open ESM-2 and extract protein representations

Just as DistilBERT produced a contextual vector for each text token, ESM-2 can produce a contextual vector for each amino-acid position.

We now load the base ESM encoder directly and inspect its hidden-state tensor.

In [ ]:
from transformers import AutoModel

esm_encoder = AutoModel.from_pretrained(
    ESM_MODEL
).to(DEVICE)

esm_encoder.eval()

protein_sequence = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)

protein_inputs = esm_tokenizer(
    protein_sequence,
    return_tensors="pt"
)

protein_inputs = {
    key: value.to(DEVICE)
    for key, value in protein_inputs.items()
}

with torch.no_grad():
    protein_outputs = esm_encoder(
        **protein_inputs,
        return_dict=True
    )

protein_hidden = protein_outputs.last_hidden_state

print("Protein length:", len(protein_sequence))
print("Hidden-state shape:", tuple(protein_hidden.shape))
print("\nRepresentation dimensions per token:", protein_hidden.shape[-1])

## What should you notice?

The output has the same broad structural idea as the text Transformer output:

- batch;
- sequence positions;
- hidden representation dimensions.

The numbers do not directly tell a biologist “this is a binding site” or “this protein has this function.”

They are **learned features**.

Downstream models can use these representations as inputs for more specific biological prediction tasks.

# 6.6 Convert an entire protein into one summary vector

Sometimes a downstream task requires one vector for an entire sequence rather than one vector per amino-acid position.

For this demonstration, we will:

1. obtain contextual representations for every residue;
2. exclude special tokens;
3. average the residue representations.

This simple mean-pooling method is useful for teaching the representation concept.

It is **not a claim that mean pooling is the optimal biological representation for every downstream task**.

In [ ]:
def get_protein_embedding(sequence):
    encoded = esm_tokenizer(
        sequence,
        return_tensors="pt"
    )

    input_ids_cpu = encoded["input_ids"][0]

    special_token_mask = esm_tokenizer.get_special_tokens_mask(
        input_ids_cpu.tolist(),
        already_has_special_tokens=True
    )

    encoded = {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = esm_encoder(
            **encoded,
            return_dict=True
        )

    hidden = outputs.last_hidden_state[0]

    residue_positions = [
        index
        for index, is_special in enumerate(special_token_mask)
        if is_special == 0
    ]

    residue_hidden = hidden[residue_positions]

    sequence_embedding = residue_hidden.mean(dim=0)

    return sequence_embedding

example_embedding = get_protein_embedding(protein_sequence)

print("Sequence embedding shape:", tuple(example_embedding.shape))
print("First 10 values:")
print(example_embedding[:10].detach().cpu())

# 6.7 A representation experiment: similar sequence vs. synthetic control

Now we ask whether ESM-2's learned representation reacts to changes in sequence.

We create three inputs:

- **Sequence A** — the original example;
- **Sequence B** — the same sequence with one amino-acid substitution;
- **Sequence C** — a synthetic control sequence with very different ordering.

We compute one embedding for each and compare them using cosine similarity.

### Important scientific caution

This is a **pedagogical representation experiment**.

Cosine similarity of mean-pooled ESM embeddings is not, by itself, a validated measure of protein function, structure, evolutionary relationship, or clinical significance.

In [ ]:
sequence_a = protein_sequence

# One controlled substitution relative to sequence A.
sequence_b = (
    protein_sequence[:24]
    + ("E" if protein_sequence[24] != "E" else "D")
    + protein_sequence[25:]
)

# Synthetic comparison sequence; not presented as a natural protein.
sequence_c = (
    "ACDEFGHIKLMNPQRSTVWY"
    "YWVTSRQPNMLKIHGFEDCA"
    "ACDEFGHIKLMNPQRSTVWY"
    "YWVTSRQPNMLKIHG"
)

embedding_a = get_protein_embedding(sequence_a)
embedding_b = get_protein_embedding(sequence_b)
embedding_c = get_protein_embedding(sequence_c)

similarity_ab = cosine_similarity(
    embedding_a.unsqueeze(0),
    embedding_b.unsqueeze(0)
).item()

similarity_ac = cosine_similarity(
    embedding_a.unsqueeze(0),
    embedding_c.unsqueeze(0)
).item()

print("Cosine similarity:")
print(f"Sequence A vs. one-substitution Sequence B: {similarity_ab:.4f}")
print(f"Sequence A vs. synthetic Sequence C:       {similarity_ac:.4f}")

## How to interpret this exercise

You should generally expect the one-substitution sequence to retain a representation closer to the original than a deliberately different control sequence.

The exact numbers are less important than the concept:

> **A scientific foundation model converts complex domain data into reusable learned representations.**

Those representations can then be used by downstream systems.

### Potential downstream directions

Protein foundation-model representations can support research tasks such as:

- protein classification;
- functional annotation;
- mutation-effect modeling;
- localization prediction;
- structure-related modeling;
- other sequence-based biological prediction problems.

Each real application requires task-specific data, validation, and biological expertise.

# 6.8 Why this matters for the meaning of “foundation model”

At this point, the workshop has used two very different domains:

**Human language**
- tokens are linguistic units;
- the Transformer learns contextual language representations.

**Protein biology**
- tokens are amino acids;
- the Transformer learns contextual biological sequence representations.

The general pattern is broader than either domain:

- large-scale pretraining;
- reusable representation;
- downstream adaptation.

This is the central reason we should not treat **foundation model** and **chatbot** as synonyms.

### Main-lecture connection

This section makes the ESM-2 example in **Section E — Biology and Medicine** directly executable.

It also reinforces **Section D — Self-supervised Pretraining** by showing that masked prediction is not restricted to natural language.

### Next step

Proteins are still sequences.

We will now move to another kind of data entirely:

> **images and multimodal representations**

before ending with a scientific Earth-observation foundation model.

# 7. Multimodal Foundation Models: Connecting Images and Text

## Why this section exists

Foundation models can also learn relationships **across different data modalities**.

A multimodal model may learn representations of:

- text;
- images;
- audio;
- video;
- or combinations of them.

We will use **CLIP** as a brief conceptual bridge.

CLIP learns image and text representations that can be compared in a shared representation space. This enables **zero-shot image classification**: we can supply new candidate text labels at inference time without training a new classifier in this notebook.

### Connection to the main lecture

This section connects to:

**Section D — Developments in Multimodal Data Representations**

and to:

**Radford et al. (2021) — CLIP**.

> **Workshop bandwidth note:** the CLIP model weights are substantially larger than the small ESM-2 checkpoint. The actual CLIP execution is therefore marked as an optional instructor-led exercise. The conceptual section remains part of the core notebook.

## Visual overview — A shared representation space

The next plain infographic shows the main idea.

An image encoder and a text encoder produce vectors. Training encourages related image-text pairs to be closer in the learned representation space.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)
ax.axis("off")

elements = [
    (0.5, 3.0, 2.2, 1.1, "IMAGE", "pixels"),
    (3.3, 3.0, 2.2, 1.1, "IMAGE ENCODER", "visual representation"),
    (0.5, 0.9, 2.2, 1.1, "TEXT", 'e.g. "a field"'),
    (3.3, 0.9, 2.2, 1.1, "TEXT ENCODER", "language representation"),
    (7.2, 1.9, 3.4, 1.2, "SHARED REPRESENTATION SPACE",
     "compare vector similarity"),
]

for x, y, w, h, title, body in elements:
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.4, fill=False
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + 0.72, title,
            ha="center", va="center",
            fontsize=10.5, fontweight="bold")
    ax.text(x + w/2, y + 0.30, body,
            ha="center", va="center", fontsize=9.5)

for y in [3.55, 1.45]:
    ax.add_patch(
        FancyArrowPatch(
            (2.75, y), (3.25, y),
            arrowstyle="->", mutation_scale=15, linewidth=1.3
        )
    )

ax.add_patch(
    FancyArrowPatch(
        (5.55, 3.55), (7.1, 2.75),
        arrowstyle="->", mutation_scale=15, linewidth=1.3
    )
)
ax.add_patch(
    FancyArrowPatch(
        (5.55, 1.45), (7.1, 2.25),
        arrowstyle="->", mutation_scale=15, linewidth=1.3
    )
)

ax.text(6.0, 4.65,
        "Multimodal learning connects representations from different data types",
        ha="center", va="center",
        fontsize=14, fontweight="bold")

plt.show()

## Cell 7.1 — Inspect CLIP without downloading its full weights

As with the previous models, we begin with the model repository.

This lets us discuss a useful discipline:

> **Inspect first; download second.**

The model card identifies the model's task and documents important limitations.

In [ ]:
CLIP_MODEL = "openai/clip-vit-base-patch32"

clip_info = model_info(CLIP_MODEL)

print("Model ID:", clip_info.id)
print("Pipeline task:", clip_info.pipeline_tag)
print("License:", clip_info.card_data.license if clip_info.card_data else "Not specified")

print("\nSelected tags:")
for tag in (clip_info.tags or [])[:12]:
    print(" -", tag)

# 7.2 Optional instructor demo — zero-shot image classification

This exercise is optional during the 90-minute session because it requires downloading a much larger model than ESM-2.

If the instructor has already cached the model or the network is reliable, set:

```python
RUN_CLIP_DEMO = True
```

The code uses a public sample image from the Hugging Face documentation and provides several text labels.

The important question is:

> **Did we train a new image classifier on these labels in this notebook?**

No. The model compares image and text representations learned during pretraining.

In [ ]:
RUN_CLIP_DEMO = False

if RUN_CLIP_DEMO:
    clip_classifier = pipeline(
        task="zero-shot-image-classification",
        model=CLIP_MODEL,
        device=0 if torch.cuda.is_available() else -1
    )

    sample_image_url = (
        "https://huggingface.co/datasets/"
        "huggingface/documentation-images/resolve/main/hub/parrots.png"
    )

    candidate_labels = [
        "animals",
        "humans",
        "landscape",
        "buildings",
        "vehicles"
    ]

    clip_results = clip_classifier(
        sample_image_url,
        candidate_labels=candidate_labels
    )

    for result in clip_results:
        print(
            f"{result['label']:<15s} "
            f"{result['score']:.4f}"
        )
else:
    print(
        "CLIP demo skipped. "
        "Set RUN_CLIP_DEMO = True for the optional instructor demonstration."
    )

## Main idea from the multimodal section

The same representation-learning philosophy can connect **different modalities**, not just different tasks within one modality.

This is a useful bridge to modern multimodal systems, but for this workshop we now return to a scientific domain with particularly clear relevance to African research:

> **Earth observation.**

# 8. Earth Observation Foundation Models: Prithvi

## Why this section exists

Satellite data is important for many research and policy problems across Africa, including:

- agriculture and crop monitoring;
- drought and vegetation dynamics;
- flood assessment;
- wildfire and burn-scar mapping;
- land-cover change;
- forest monitoring;
- water resources;
- urban expansion.

Earth-observation data is fundamentally different from ordinary photographs.

It may contain:

- multiple spectral bands beyond visible RGB;
- repeated observations over time;
- large spatial coverage;
- geospatial metadata.

We will examine the **IBM–NASA Prithvi Earth Observation foundation model**.

### Connection to the main lecture

This section connects directly to:

**Section E — Earth, Climate and Geospatial Science**

and:

**Jakubik et al. (2023) — Prithvi / geospatial foundation models**.

## Visual overview — What enters an Earth-observation foundation model?

The next plain infographic emphasizes two aspects that distinguish this example from ordinary image classification:

1. **multiple spectral bands**;
2. **multiple time steps**.

The model converts spatiotemporal satellite observations into reusable representations that can later support downstream geospatial tasks.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(0, 13)
ax.set_ylim(0, 5)
ax.axis("off")

# Input stack
input_box = FancyBboxPatch(
    (0.4, 1.35), 3.0, 2.3,
    boxstyle="round,pad=0.05,rounding_size=0.08",
    linewidth=1.5, fill=False
)
ax.add_patch(input_box)
ax.text(1.9, 3.20, "SATELLITE TIME SERIES",
        ha="center", fontsize=11, fontweight="bold")
ax.text(1.9, 2.72, "Blue • Green • Red", ha="center", fontsize=9.5)
ax.text(1.9, 2.35, "NIR • SWIR 1 • SWIR 2", ha="center", fontsize=9.5)
ax.text(1.9, 1.88, "Time 1 • Time 2 • Time 3", ha="center", fontsize=9.5)
ax.text(1.9, 1.52, "spatial × spectral × temporal", ha="center", fontsize=9.2)

patch_box = FancyBboxPatch(
    (4.4, 1.65), 2.2, 1.7,
    boxstyle="round,pad=0.05,rounding_size=0.08",
    linewidth=1.5, fill=False
)
ax.add_patch(patch_box)
ax.text(5.5, 2.75, "PATCHES", ha="center",
        fontsize=11, fontweight="bold")
ax.text(5.5, 2.30, "small spatial +", ha="center", fontsize=9.5)
ax.text(5.5, 1.98, "temporal units", ha="center", fontsize=9.5)

model_box = FancyBboxPatch(
    (7.5, 1.65), 2.2, 1.7,
    boxstyle="round,pad=0.05,rounding_size=0.08",
    linewidth=1.5, fill=False
)
ax.add_patch(model_box)
ax.text(8.6, 2.75, "PRITHVI", ha="center",
        fontsize=11, fontweight="bold")
ax.text(8.6, 2.30, "temporal Vision", ha="center", fontsize=9.5)
ax.text(8.6, 1.98, "Transformer", ha="center", fontsize=9.5)

output_box = FancyBboxPatch(
    (10.6, 1.35), 2.0, 2.3,
    boxstyle="round,pad=0.05,rounding_size=0.08",
    linewidth=1.5, fill=False
)
ax.add_patch(output_box)
ax.text(11.6, 3.18, "DOWNSTREAM", ha="center",
        fontsize=10.5, fontweight="bold")
ax.text(11.6, 2.72, "flood", ha="center", fontsize=9.5)
ax.text(11.6, 2.37, "land cover", ha="center", fontsize=9.5)
ax.text(11.6, 2.02, "crop mapping", ha="center", fontsize=9.5)
ax.text(11.6, 1.67, "burn scars", ha="center", fontsize=9.5)

for x1, x2 in [(3.45, 4.35), (6.65, 7.45), (9.75, 10.55)]:
    ax.add_patch(
        FancyArrowPatch(
            (x1, 2.5), (x2, 2.5),
            arrowstyle="->", mutation_scale=16, linewidth=1.4
        )
    )

ax.text(6.5, 4.55,
        "Earth-observation foundation models learn reusable spatiotemporal representations",
        ha="center", fontsize=14, fontweight="bold")

plt.show()

# 8.1 Inspect the Prithvi model repository

We first inspect the model metadata without downloading the large model weights.

### What to pay attention to

Unlike our earlier text model, the repository is tagged for:

- geospatial data;
- temporal vision;
- Vision Transformer-style processing.

Also notice the license. Model licensing is part of responsible research and deployment.

In [ ]:
PRITHVI_MODEL = "ibm-nasa-geospatial/Prithvi-EO-1.0-100M"

prithvi_info = model_info(PRITHVI_MODEL)

print("Model ID:", prithvi_info.id)
print("Pipeline task:", prithvi_info.pipeline_tag)
print("License:", prithvi_info.card_data.license if prithvi_info.card_data else "Not specified")

print("\nSelected tags:")
for tag in (prithvi_info.tags or [])[:15]:
    print(" -", tag)

# 8.2 Read the model configuration without loading the weights

The Prithvi repository includes a configuration describing the data geometry expected by the pretrained model.

Instead of loading hundreds of megabytes of weights merely to inspect these properties, we download the small configuration file.

This demonstrates another practical Hugging Face skill: sometimes **metadata is enough to answer an engineering question**.

In [ ]:
import json
from huggingface_hub import hf_hub_download

prithvi_config_path = hf_hub_download(
    repo_id=PRITHVI_MODEL,
    filename="config.json"
)

with open(prithvi_config_path, "r") as file:
    prithvi_config = json.load(file)

pretrained_cfg = prithvi_config["pretrained_cfg"]

print("Image size:", pretrained_cfg["img_size"])
print("Patch size:", pretrained_cfg["patch_size"])
print("Number of time frames:", pretrained_cfg["num_frames"])
print("Number of input channels:", pretrained_cfg["in_chans"])
print("Embedding dimension:", pretrained_cfg["embed_dim"])
print("Transformer depth:", pretrained_cfg["depth"])
print("Attention heads:", pretrained_cfg["num_heads"])
print("Pretraining mask ratio:", pretrained_cfg["mask_ratio"])

print("\nSpectral bands:")
for band in pretrained_cfg["bands"]:
    print(" -", band)

## What should you notice?

This is not an ordinary three-channel RGB image model.

The configuration expects six Earth-observation spectral channels:

- Blue;
- Green;
- Red;
- Narrow Near-Infrared;
- Short-Wave Infrared 1;
- Short-Wave Infrared 2.

The pretrained configuration also includes multiple temporal frames.

This means that **data representation is domain-specific** even when the underlying model architecture is Transformer-based.

# 8.3 How many patches does one configured input contain?

The pretrained configuration uses:

- image size: 224 × 224;
- spatial patch size: 16 × 16;
- three time frames;
- temporal patch size: 1.

We can calculate how many spatiotemporal patches are formed before Transformer processing.

In [ ]:
image_size = pretrained_cfg["img_size"]
temporal_patch, patch_height, patch_width = pretrained_cfg["patch_size"]
num_frames = pretrained_cfg["num_frames"]

patches_across_height = image_size // patch_height
patches_across_width = image_size // patch_width
patches_across_time = num_frames // temporal_patch

total_patches = (
    patches_across_height
    * patches_across_width
    * patches_across_time
)

print("Patches across image height:", patches_across_height)
print("Patches across image width:", patches_across_width)
print("Patches across time:", patches_across_time)
print("Total spatiotemporal patches:", total_patches)

## Why this matters

Earlier:

- DistilBERT processed linguistic tokens;
- ESM-2 processed amino-acid tokens.

Here, the model processes **spatiotemporal patches** derived from multispectral satellite observations.

The conceptual pattern remains recognizable:

- divide complex data into model units;
- learn contextual relationships;
- produce reusable representations.

But the domain changes the meaning of those units.

# 8.4 A critical Africa-wide question: where was the model pretrained?

The Prithvi model card states that this version was pretrained using Harmonized Landsat Sentinel-2 data from the **contiguous United States**.

This creates an excellent responsible-deployment question for an African audience.

A model can be technically open and globally accessible while its pretraining geography is still limited.

### Discuss before proceeding

Suppose you want to use the model for:

- Sahelian dryland agriculture;
- tropical forest monitoring in Central Africa;
- irrigated agriculture in North Africa;
- coastal flood analysis;
- highland crop systems;
- rapidly growing African cities.

Would pretrained weights from another geographic distribution automatically be sufficient?

Not necessarily.

You may need:

- local evaluation;
- regionally representative data;
- fine-tuning;
- calibration;
- domain-expert validation;
- comparison with simpler baselines.

This is a concrete example of the lecture's broader point:

> **Foundation models reduce the cost of starting, but they do not remove the need for local scientific validation.**

# 8.5 Optional instructor step — load the Prithvi weights

The core 90-minute participant path stops at configuration inspection because downloading this model simultaneously across a workshop room may consume substantial bandwidth.

If the instructor has pre-cached the weights, set:

```python
RUN_PRITHVI_LOAD = True
```

This optional cell loads the model and counts its parameters.

We intentionally do **not** run meaningless inference on random synthetic satellite values. Scientific input should respect the model's expected bands, units, preprocessing, and temporal structure.

In [ ]:
RUN_PRITHVI_LOAD = False

if RUN_PRITHVI_LOAD:
    from transformers import AutoModel

    # Release the protein model first if GPU memory is limited.
    if "esm_encoder" in globals():
        del esm_encoder
    if "protein_fill_mask" in globals():
        del protein_fill_mask

    cleanup_memory()

    prithvi_model = AutoModel.from_pretrained(
        PRITHVI_MODEL,
        device_map="auto"
    )

    prithvi_parameters = sum(
        parameter.numel()
        for parameter in prithvi_model.parameters()
    )

    print(
        f"Prithvi parameters: "
        f"{prithvi_parameters / 1e6:.1f} million"
    )

    show_gpu_memory()
else:
    print(
        "Prithvi weight download skipped. "
        "Set RUN_PRITHVI_LOAD = True for the optional instructor demonstration."
    )

## Main idea from the Earth-observation section

Prithvi demonstrates that foundation-model design can operate over:

- space;
- spectral channels;
- time.

It also gives us an important lesson for Africa:

> **Access to a global pretrained model is valuable, but suitability for a local ecological, agricultural, climatic, or geographic context must be demonstrated rather than assumed.**

In the final section, we will bring the entire workshop together and ask how an African research team should decide whether to **use, adapt, or reject** a foundation model.

# 9. From Pretrained Model to Responsible African Application

## Why this section exists

A foundation model is not a finished application.

The workshop has shown that pretrained models can dramatically reduce the amount of work needed to begin a new AI project.

But reuse creates a new engineering and governance responsibility:

> **We must determine whether the pretrained model is appropriate for the problem, population, scientific domain, data, infrastructure, and institution in which it will be used.**

This is particularly important when models are transferred across:

- languages;
- countries;
- ecological zones;
- healthcare systems;
- research populations;
- satellite domains;
- regulatory environments.

### Connection to the main lecture

This section connects directly to:

**Section F — Synthesis & Responsible Deployment**

including:

- bias and representation gaps;
- local and regional data;
- energy and infrastructure constraints;
- data sovereignty;
- equitable compute access;
- locally grounded validation.

## Visual framework — Use, adapt, or reject?

The following plain infographic presents a practical decision process.

The most important idea is that finding a model on the Hugging Face Hub is only the beginning of the evaluation process.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13.5, 5.2))
ax.set_xlim(0, 13.5)
ax.set_ylim(0, 5.2)
ax.axis("off")

steps = [
    (0.3, 1.8, 2.0, 1.4, "1. DEFINE", "problem + users\nscientific objective"),
    (2.9, 1.8, 2.0, 1.4, "2. INSPECT", "model card\ndata + license"),
    (5.5, 1.8, 2.0, 1.4, "3. TEST", "local data\nbaselines + failures"),
    (8.1, 1.8, 2.0, 1.4, "4. DECIDE", "use as-is\nadapt / reject"),
    (10.7, 1.8, 2.4, 1.4, "5. GOVERN", "monitoring\nhuman oversight"),
]

for x, y, w, h, title, body in steps:
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.05,rounding_size=0.08",
        linewidth=1.5,
        fill=False
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + 0.95, title,
            ha="center", va="center",
            fontsize=10.5, fontweight="bold")
    ax.text(x + w/2, y + 0.43, body,
            ha="center", va="center", fontsize=9.5)

for start_x, end_x in [(2.35, 2.85), (4.95, 5.45), (7.55, 8.05), (10.15, 10.65)]:
    ax.add_patch(
        FancyArrowPatch(
            (start_x, 2.5), (end_x, 2.5),
            arrowstyle="->", mutation_scale=16, linewidth=1.4
        )
    )

ax.text(
    6.75, 4.45,
    "A pretrained foundation model must earn its place in a real system",
    ha="center", va="center",
    fontsize=14, fontweight="bold"
)

ax.text(
    6.75, 0.75,
    "Capability alone is not enough: context, evidence, infrastructure, and governance matter.",
    ha="center", va="center", fontsize=11
)

plt.show()

# 9.1 Team exercise — Three African research scenarios

Divide participants into small groups.

Each group receives one scenario.

The goal is **not** to design the full technical system.

The goal is to decide what evidence would be needed before reusing a foundation model.

---

## Scenario A — Agriculture and Earth Observation

A regional research consortium wants to use satellite imagery to identify crop stress across several agroecological zones.

A pretrained geospatial foundation model is available.

Discuss:

- Was its pretraining geography representative of your target regions?
- Are the required satellite bands and temporal observations available?
- What locally labeled data would be needed for evaluation?
- Should the model be used as-is or fine-tuned?
- How will performance be compared across agroecological zones?
- What happens if cloud cover, sensor differences, or seasonal patterns differ from the pretraining data?

---

## Scenario B — Biology and Health Research

A research group wants to use protein foundation-model representations to prioritize candidate proteins for further laboratory investigation.

Discuss:

- What biological task are you actually predicting?
- Is the pretrained representation sufficient, or is a downstream model required?
- What labeled biological data are available?
- How will laboratory/domain experts validate the predictions?
- What failure could occur if embedding similarity is mistaken for biological equivalence?
- Which claims can the AI system support, and which require experimental evidence?

---

## Scenario C — Multilingual Public Information

A public institution wants to build an information system that serves users across several African languages.

Discuss:

- Which languages and dialects are supported?
- How well does the tokenizer represent them?
- What evaluation data exist for each language?
- Are some languages receiving substantially poorer results than others?
- Where will prompts and user data be processed?
- What human review is needed for high-impact information?

# 9.2 A reusable model-evaluation checklist

Before deploying or adapting a foundation model, examine at least the following dimensions.

| Dimension | Core question |
|---|---|
| **Task fit** | Was the model designed or evaluated for the problem we are solving? |
| **Domain fit** | Does the pretraining data represent our scientific or operational domain? |
| **Geographic fit** | Does the evidence cover the regions where the system will be used? |
| **Language fit** | Are the relevant languages and writing systems represented adequately? |
| **Local data** | What local data are required for validation or adaptation? |
| **Benchmarking** | Does the model outperform appropriate simpler baselines? |
| **Failure modes** | How does the system fail, and who bears the consequences? |
| **Human expertise** | Where must domain experts remain in the loop? |
| **Compute** | Can the institution afford training and inference reliably? |
| **Energy** | Is the compute strategy proportionate to the problem? |
| **Privacy** | What sensitive or proprietary information enters the system? |
| **Sovereignty** | Where are data, models, and inference infrastructure controlled? |
| **License** | Does the license permit the intended research or deployment? |
| **Monitoring** | How will performance be re-evaluated after deployment? |

A technically impressive model can still be the wrong model for a particular application.

# 10. What Did We Actually Learn?

The workshop began with a simple text classifier, but the objective was never to teach only natural-language processing.

We progressively uncovered the general foundation-model pattern.

## We used

A pretrained model through a high-level Hugging Face interface.

## We opened

The model pipeline and inspected:

- tokens;
- token IDs;
- attention masks;
- contextual representations;
- self-attention.

## We adapted

A pretrained Transformer using a small labeled dataset and measured the actual fine-tuning time.

## We crossed domains

We used ESM-2 to show that:

- a token can be an amino acid;
- masked prediction can be a biological pretraining objective;
- Transformers can produce reusable protein representations.

## We crossed modalities

We examined how image and text representations can occupy a shared learned space.

## We entered Earth observation

We examined how Prithvi represents:

- spectral information;
- space;
- time;
- satellite image patches.

## We returned to deployment

We asked whether a globally available model is actually appropriate for a specific African scientific, linguistic, geographic, or institutional context.

## Final visual synthesis — One paradigm, many domains

The final infographic summarizes the conceptual argument of the entire lab.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(13.5, 7.2))
ax.set_xlim(0, 13.5)
ax.set_ylim(0, 7.2)
ax.axis("off")

# Central model
center = FancyBboxPatch(
    (4.7, 2.7), 4.1, 1.7,
    boxstyle="round,pad=0.06,rounding_size=0.1",
    linewidth=1.8, fill=False
)
ax.add_patch(center)
ax.text(6.75, 3.75, "FOUNDATION-MODEL PARADIGM",
        ha="center", fontsize=12, fontweight="bold")
ax.text(6.75, 3.32, "large-scale pretraining",
        ha="center", fontsize=10)
ax.text(6.75, 2.98, "reusable learned representations",
        ha="center", fontsize=10)

domains = [
    (0.35, 5.35, 2.6, 1.2, "LANGUAGE", "words / subwords"),
    (3.55, 5.35, 2.6, 1.2, "BIOLOGY", "amino acids"),
    (6.75, 5.35, 2.6, 1.2, "MULTIMODAL", "image + text"),
    (9.95, 5.35, 2.9, 1.2, "EARTH OBSERVATION", "spectral × space × time"),
]

for x, y, w, h, title, body in domains:
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.4, fill=False
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + 0.78, title,
            ha="center", fontsize=10.5, fontweight="bold")
    ax.text(x + w/2, y + 0.32, body,
            ha="center", fontsize=9.5)

for x in [1.65, 4.85, 8.05, 11.4]:
    ax.add_patch(
        FancyArrowPatch(
            (x, 5.3), (6.75, 4.45),
            arrowstyle="->", mutation_scale=14, linewidth=1.1
        )
    )

application = FancyBboxPatch(
    (3.2, 0.55), 7.1, 1.2,
    boxstyle="round,pad=0.05,rounding_size=0.08",
    linewidth=1.5, fill=False
)
ax.add_patch(application)
ax.text(
    6.75, 1.28,
    "LOCAL APPLICATION = adaptation + validation + infrastructure + governance",
    ha="center", fontsize=10.5, fontweight="bold"
)
ax.text(
    6.75, 0.88,
    "The pretrained model is a foundation, not the finished system.",
    ha="center", fontsize=10
)

ax.add_patch(
    FancyArrowPatch(
        (6.75, 2.65), (6.75, 1.8),
        arrowstyle="->", mutation_scale=17, linewidth=1.5
    )
)

ax.set_title(
    "Foundation models are a reusable learning paradigm, not a synonym for chatbots",
    fontsize=15, pad=10
)

plt.show()

# 11. The 90-Minute Core Route

The notebook contains more material than should be executed by every participant during the live session.

Use the following route for the actual 90-minute workshop.

| Time | Core activity | Approx. duration |
|---|---|---:|
| 0–7 min | Environment and GPU inspection | 7 min |
| 7–14 min | Hugging Face Hub + first pretrained model | 7 min |
| 14–27 min | Tokenization + multilingual representation experiment | 13 min |
| 27–37 min | Contextual representations + attention | 10 min |
| 37–52 min | Pretraining intuition + fine-tuning | 15 min |
| 52–69 min | ESM-2 protein foundation model | 17 min |
| 69–75 min | Protein embedding similarity | 6 min |
| 75–80 min | Multimodal concept / optional CLIP | 5 min |
| 80–86 min | Prithvi configuration and Earth-observation discussion | 6 min |
| 86–90 min | Responsible deployment + final synthesis | 4 min |

## Optional cells during the live session

To protect the schedule and workshop bandwidth:

- **CLIP weight download** — optional instructor demonstration;
- **Prithvi weight download** — optional instructor demonstration;
- additional tokenizer comparisons — participant extension;
- repeated attention-head experiments — participant extension;
- multiple fine-tuning epochs — post-workshop extension.

The notebook remains useful after the session because participants can return to these optional exercises independently.

# 12. Suggested Post-Workshop Investigations

## A. Compare African-language tokenizers

Select one or more languages relevant to your work.

Compare:

- an English-focused tokenizer;
- a multilingual tokenizer;
- a language- or region-specific tokenizer if available.

Measure:

- token fragmentation;
- unknown tokens;
- sequence length;
- downstream task performance.

Do not infer model quality from token count alone.

---

## B. Full fine-tuning vs. partial fine-tuning

Repeat the DistilBERT experiment with all layers trainable.

Compare:

- trainable parameters;
- wall-clock time;
- GPU memory;
- validation performance.

---

## C. Parameter-efficient fine-tuning

Explore a method such as LoRA or another PEFT approach.

Ask:

> How much task adaptation can we achieve while updating only a small fraction of parameters?

---

## D. Protein downstream task

Choose a real, curated biological dataset and investigate how ESM-2 representations can be used in a supervised downstream model.

This requires biological interpretation and appropriate validation.

---

## E. Local Earth-observation evaluation

Select an open satellite dataset from a region relevant to your work.

Before fine-tuning Prithvi or another geospatial foundation model:

- inspect sensor compatibility;
- verify spectral bands;
- check preprocessing units;
- identify temporal requirements;
- construct a regionally representative evaluation set;
- establish a simpler baseline.

---

## F. Model-card audit

Choose one model on the Hugging Face Hub and prepare a short audit covering:

- intended use;
- architecture;
- pretraining data;
- evaluation;
- license;
- compute requirements;
- limitations;
- suitability for your intended context.

# 13. References Connected to the Lab

The notebook is a practical companion to the main foundation-model lecture.

## Foundational architecture and training

1. Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). *Learning representations by back-propagating errors.*
2. Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). *ImageNet Classification with Deep Convolutional Neural Networks.*
3. Mikolov, T. et al. (2013). *Efficient Estimation of Word Representations in Vector Space.*
4. Bahdanau, D., Cho, K., & Bengio, Y. (2014). *Neural Machine Translation by Jointly Learning to Align and Translate.*
5. Vaswani, A. et al. (2017). *Attention Is All You Need.*
6. Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.*
7. Wolf, T. et al. (2020). *Transformers: State-of-the-Art Natural Language Processing.*
8. Bommasani, R. et al. (2021). *On the Opportunities and Risks of Foundation Models.*

## Multimodal and scientific foundation models

9. Radford, A. et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.*
10. Jumper, J. et al. (2021). *Highly accurate protein structure prediction with AlphaFold.*
11. Lin, Z. et al. (2023). *Evolutionary-scale prediction of atomic-level protein structure with a language model.*
12. Jakubik, J. et al. (2023). *Foundation Models for Generalist Geospatial Artificial Intelligence.*
13. Lam, R. et al. (2023). *Learning skillful medium-range global weather forecasting.*
14. Merchant, A. et al. (2023). *Scaling deep learning for materials discovery.*

## Hugging Face resources used

- Transformers documentation: https://huggingface.co/docs/transformers/
- Datasets documentation: https://huggingface.co/docs/datasets/
- DistilBERT base: https://huggingface.co/distilbert/distilbert-base-uncased
- DistilBERT SST-2: https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
- XLM-RoBERTa base: https://huggingface.co/FacebookAI/xlm-roberta-base
- GLUE / SST-2: https://huggingface.co/datasets/nyu-mll/glue
- ESM-2 8M: https://huggingface.co/facebook/esm2_t6_8M_UR50D
- CLIP ViT-B/32: https://huggingface.co/openai/clip-vit-base-patch32
- Prithvi-EO 100M: https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-1.0-100M

# 14. Instructor Preparation Checklist

Before publishing this notebook to GitHub and using it live:

1. Open it in a **fresh Google Colab GPU runtime**.
2. Run every **core** cell from top to bottom.
3. Record the package versions printed in Section 0.
4. If desired, pin those package versions in the installation cell.
5. Confirm the DistilBERT fine-tuning wall-clock time on the available Colab GPU.
6. Confirm that the ESM-2 checkpoint downloads and runs.
7. Decide in advance whether to enable the optional CLIP demonstration.
8. Pre-cache CLIP if you plan to run it live.
9. Decide whether the Prithvi section will remain metadata-only or whether the weights will be pre-cached for instructor inspection.
10. Keep one **fully executed instructor copy** of the notebook available as a fallback.
11. Ask participants to bring a Google account capable of opening Colab.
12. Avoid relying on every participant receiving the same GPU type.

## GitHub structure

A simple repository can contain:

```text
foundation-models-workshop/
├── README.md
└── Foundation_Models_in_Practice_Africa_Workshop_v2.ipynb
```

After uploading the notebook, add an **Open in Colab** badge to the repository README and, if desired, to the top of this notebook.

# Thank You

## Final takeaway

> **Foundation models are a general-purpose approach to learning reusable representations from large-scale data.**

The data may be:

- human language;
- protein sequences;
- images;
- satellite observations;
- or other scientific modalities.

The pretrained model is only the foundation.

Real benefit comes from combining it with:

- appropriate local or domain data;
- scientific and engineering expertise;
- accessible compute;
- careful evaluation;
- responsible governance.

**Questions and discussion**